# Similar Item Recommendation (RetailRocket Dataset)

This notebook demonstrates item-to-item recommendation using embedding-based retrieval.

**Dataset**  
- RetailRocket: `events.csv`, `item_properties_part1.csv`, `item_properties_part2.csv`

**Approach**  
1. Extract "view" events and build item sequences per user session.  
2. Train item embeddings (Item2Vec style) to capture product similarity.  
3. Build an ANN (Approximate Nearest Neighbor) index for fast retrieval.  

**Output**  
- Given a seed item, retrieve the most similar items based on user interaction patterns.

In [1]:
rootpath = "./Data/"

In [2]:
import pandas as pd

In [3]:
events = pd.read_csv("../Data/events.csv")
print(events.head())

       timestamp  visitorid event  itemid  transactionid
0  1433221332117     257597  view  355908            NaN
1  1433224214164     992329  view  248676            NaN
2  1433221999827     111016  view  318965            NaN
3  1433221955914     483717  view  253185            NaN
4  1433221337106     951259  view  367447            NaN


In [4]:
# Filter only 'view' events
view_events = events[events["event"] == "view"].copy()

# Add a new column 'hour' (timestamp in hours)
view_events["hour"] = view_events["timestamp"] // (1000 * 60 * 60)

# Show the result
view_events.head()

,timestamp,visitorid,event,itemid,transactionid,hour
0,1433221332117,257597,view,355908,NaN,398117
1,1433224214164,992329,view,248676,NaN,398117
2,1433221999827,111016,view,318965,NaN,398117
3,1433221955914,483717,view,253185,NaN,398117
4,1433221337106,951259,view,367447,NaN,398117


In [5]:
view_events.groupby(['visitorid', 'hour']).count()['itemid'].mean()

1.4843617072840531

In [6]:
ids = view_events.groupby(['visitorid', 'hour']).count()
print(ids[ids.itemid > 1]['itemid'].count())
print(ids[ids.itemid > 1]['itemid'].mean())

373448
3.3280108609498513


In [7]:
view_events.groupby(['visitorid']).count()['itemid'].mean()

1.8974162126053729

In [8]:
ids = view_events.groupby(['visitorid']).count()
print(ids[ids.itemid > 1]['itemid'].count())
print(ids[ids.itemid > 1]['itemid'].mean())

396449
4.1785500783202885


In [9]:
groups = view_events.groupby('visitorid')['itemid'].apply(list)

In [10]:
groups = groups.reset_index()

In [11]:
groups

,visitorid,itemid
0,0,"[285930, 357564, 67045]"
1,1,[72028]
2,2,"[216305, 325215, 342816, 325215, 342816, 25988..."
3,3,[385090]
4,4,[177677]
...,...,...
1404174,1407575,[121220]
1404175,1407576,[356208]
1404176,1407577,[427784]
1404177,1407578,[188736]


In [12]:
data = groups[groups['itemid'].map(len) > 1]['itemid']

In [13]:
data_list = data.to_list()

In [14]:
data_list[0]

[285930, 357564, 67045]

In [15]:
# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random

In [16]:
# Build vocabulary
item2idx = {item: idx for idx, item in enumerate(set(i for s in data_list for i in s))}
idx2item = {v: k for k, v in item2idx.items()}

# Generate skip-gram pairs
def generate_pairs(sessions, window=2):
    pairs = []
    for session in sessions:
        idxs = [item2idx[i] for i in session]
        for c in range(len(idxs)):
            for w in range(-window, window+1):
                p = c + w
                if p < 0 or p >= len(idxs) or p == c:
                    continue
                pairs.append((idxs[c], idxs[p]))
    return pairs

pairs = generate_pairs(data_list)

# Sample 10% of pairs for speed
random.shuffle(pairs)
pairs = pairs[:len(pairs)//10]

# Define model
class Item2Vec(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, dim)
    def forward(self, c, o):
        return (self.emb(c) * self.emb(o)).sum(1)

model = Item2Vec(len(item2idx), 32)

# Mini-batch training
opt = optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()
batch_size = 1024

for i in range(0, len(pairs), batch_size):
    batch = pairs[i:i+batch_size]
    c = torch.tensor([p[0] for p in batch])
    o = torch.tensor([p[1] for p in batch])
    opt.zero_grad()
    out = model(c, o)
    loss = loss_fn(out, torch.ones_like(out))
    loss.backward()
    opt.step()
print("training done (sampled, batched)")


training done (sampled, batched)


In [17]:
import os
os.makedirs(rootpath, exist_ok=True)

# Save model weights
torch.save(model.state_dict(), rootpath + "item2vec.pt")
print("saved:", rootpath + "item2vec.pt")

saved: ./Data/item2vec.pt


In [18]:
# Prepare embeddings
emb = model.emb.weight.detach().numpy()

# Cosine similarity based most_similar
def most_similar(item_id, topn=5):
    if item_id not in item2idx:
        return []
    vec = emb[item2idx[item_id]]
    sims = emb @ vec / (np.linalg.norm(emb, axis=1) * (np.linalg.norm(vec) + 1e-9))
    order = np.argsort(-sims)
    order = [i for i in order if i != item2idx[item_id]][:topn]
    return [(idx2item[i], float(sims[i])) for i in order]

print(most_similar(342816))

[(8566, 0.7039357423782349), (252986, 0.6788788437843323), (148971, 0.6781764030456543), (196905, 0.6773132085800171), (30334, 0.6534613370895386)]


In [19]:
# Save embeddings in word2vec text format
with open(rootpath + "item_vector.txt", "w") as f:
    f.write(f"{len(item2idx)} 32\n")
    for idx, vec in enumerate(emb):
        f.write(f"{idx2item[idx]} " + " ".join(map(str, vec.tolist())) + "\n")
print("saved: item_vector.txt")

saved: item_vector.txt
